In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from zoneinfo import ZoneInfo

=====================================
# Step 2: Load the Dataset
=====================================

Load the Google Play Store dataset into a Pandas DataFrame
for further data cleaning and visualization.

In [15]:
play_store_data = pd.read_csv(
    r"C:\Users\Vansh Sharma\Downloads\Play Store Data (1).csv"
)

print(play_store_data.head())

                                                 App        Category  Rating  \
0     Photo Editor & Candy Camera & Grid & ScrapBook  ART_AND_DESIGN     4.1   
1                                Coloring book moana  ART_AND_DESIGN     3.9   
2  U Launcher Lite – FREE Live Cool Themes, Hide ...  ART_AND_DESIGN     4.7   
3                              Sketch - Draw & Paint  ART_AND_DESIGN     4.5   
4              Pixel Draw - Number Art Coloring Book  ART_AND_DESIGN     4.3   

  Reviews  Size     Installs  Type Price Content Rating  \
0     159   19M      10,000+  Free     0       Everyone   
1     967   14M     500,000+  Free     0       Everyone   
2   87510  8.7M   5,000,000+  Free     0       Everyone   
3  215644   25M  50,000,000+  Free     0           Teen   
4     967  2.8M     100,000+  Free     0       Everyone   

                      Genres      Last Updated         Current Ver  \
0               Art & Design   January 7, 2018               1.0.0   
1  Art & Design;Pretend 

=====================================
# Step 3: Explore the Dataset
=====================================

Understand the structure of the dataset by examining
its dimensions, column names, data types,
missing values, and duplicate records.

In [16]:
# Display dataset dimensions
print("Dataset Shape:", play_store_data.shape)

# Display column names
print("\nColumns:")
print(play_store_data.columns.tolist())

# Display dataset information
print("\nDataset Information:")
play_store_data.info()

# Check missing values
print("\nMissing Values:")
print(play_store_data.isnull().sum())

# Check duplicate rows
print("\nDuplicate Rows:", play_store_data.duplicated().sum())

Dataset Shape: (10841, 13)

Columns:
['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver']

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  str    
 1   Category        10841 non-null  str    
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  str    
 4   Size            10841 non-null  str    
 5   Installs        10841 non-null  str    
 6   Type            10840 non-null  str    
 7   Price           10841 non-null  str    
 8   Content Rating  10840 non-null  str    
 9   Genres          10841 non-null  str    
 10  Last Updated    10841 non-null  str    
 11  Current Ver     10833 non-null  str    
 12  Android Ver     10838 non-null  str    
dtypes: float64(

=====================================
# Step 4: Clean the Dataset
=====================================

Clean the dataset by removing duplicate records,
handling missing values, and converting important
columns into appropriate data types.

In [17]:
# Remove duplicate applications
play_store_data = play_store_data.drop_duplicates(subset="App")

# Convert Rating into numeric
play_store_data["Rating"] = pd.to_numeric(
    play_store_data["Rating"],
    errors="coerce"
)

# Convert Reviews into numeric
play_store_data["Reviews"] = pd.to_numeric(
    play_store_data["Reviews"],
    errors="coerce"
)

# Clean Installs column
play_store_data["Installs"] = (
    play_store_data["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

play_store_data["Installs"] = pd.to_numeric(
    play_store_data["Installs"],
    errors="coerce"
)

# Convert Size into MB
def convert_size(size):

    if pd.isna(size):
        return np.nan

    size = str(size)

    if size.endswith("M"):
        return float(size[:-1])

    elif size.endswith("k"):
        return float(size[:-1]) / 1024

    else:
        return np.nan


play_store_data["Size_MB"] = play_store_data["Size"].apply(convert_size)

# Remove rows having missing values
play_store_data = play_store_data.dropna(
    subset=["Rating", "Reviews", "Installs", "Size_MB", "Last Updated"]
)

play_store_data.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159.0,19M,10000.0,Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up,19.0
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14M,500000.0,Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,14.0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510.0,8.7M,5000000.0,Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up,8.7
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644.0,25M,50000000.0,Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up,25.0
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967.0,2.8M,100000.0,Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up,2.8


=====================================
# Step 5: Convert the Last Updated Column
=====================================

Convert the Last Updated column into datetime format
and extract the month to analyze cumulative installs
over time.

In [18]:
# Convert Last Updated into datetime
play_store_data["Last Updated"] = pd.to_datetime(
    play_store_data["Last Updated"],
    errors="coerce"
)

# Remove invalid dates
play_store_data = play_store_data.dropna(subset=["Last Updated"])

# Extract Month Name
play_store_data["Month"] = play_store_data["Last Updated"].dt.strftime("%b")

# Display sample
play_store_data[["Last Updated", "Month"]].head()

,Last Updated,Month
0,2018-01-07,Jan
1,2018-01-15,Jan
2,2018-08-01,Aug
3,2018-06-08,Jun
4,2018-06-20,Jun


=====================================
# Step 6: Apply Required Filters
=====================================

Filter the dataset according to the specified business requirements.

Applied Conditions:

• Average Rating ≥ 4.2

• App Name should not contain any numeric values

• Categories should start with the letters "T" or "P"

• Reviews > 1000

• App Size between 20 MB and 80 MB

In [19]:
# Keep categories starting with T or P
filtered_data = play_store_data[
    play_store_data["Category"].str.startswith(("T", "P"), na=False)
].copy()

# Apply all filters
filtered_data = filtered_data[
    (filtered_data["Rating"] >= 4.2) &
    (filtered_data["Reviews"] > 1000) &
    (filtered_data["Size_MB"].between(20, 80)) &
    (~filtered_data["App"].str.contains(r"\d", regex=True, na=False))
]

print("Filtered Dataset Shape :", filtered_data.shape)

filtered_data.head()

Filtered Dataset Shape : (110, 15)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB,Month
2802,"Shutterfly: Free Prints, Photo Books, Cards, G...",PHOTOGRAPHY,4.6,98716.0,59M,5000000.0,Free,0,Everyone,Photography,2018-08-01,5.13.1,5.0 and up,59.0,Aug
2803,FreePrints – Free Photos Delivered,PHOTOGRAPHY,4.8,109500.0,37M,1000000.0,Free,0,Everyone,Photography,2018-08-02,2.18.2,4.1 and up,37.0,Aug
2811,"Face Filter, Selfie Editor - Sweet Camera",PHOTOGRAPHY,4.7,142634.0,22M,10000000.0,Free,0,Everyone,Photography,2018-07-06,1.5.1,4.4 and up,22.0,Jul
2822,Makeup Editor -Beauty Photo Editor & Selfie Ca...,PHOTOGRAPHY,4.5,3378.0,30M,1000000.0,Free,0,Mature 17+,Photography,2018-07-25,1.7,4.1 and up,30.0,Jul
2823,Makeup Photo Editor: Makeup Camera & Makeup Ed...,PHOTOGRAPHY,4.4,10525.0,25M,1000000.0,Free,0,Everyone,Photography,2018-07-27,8.9.9,4.0 and up,25.0,Jul


=====================================
# Step 7: Translate Category Names
=====================================

Translate selected category names for visualization
purposes only.

Translations:

• Travel & Local → Voyage et Local (French)

• Productivity → Productividad (Spanish)

• Photography → 写真 (Japanese)

The original dataset remains unchanged.

In [20]:
# Category translation dictionary
translation = {
    "TRAVEL_AND_LOCAL": "Voyage et Local",
    "PRODUCTIVITY": "Productividad",
    "PHOTOGRAPHY": "写真"
}

# Create a new display column
filtered_data["Category_Display"] = (
    filtered_data["Category"]
    .replace(translation)
)

filtered_data[
    ["Category", "Category_Display"]
].drop_duplicates()

,Category,Category_Display
2802,PHOTOGRAPHY,写真
3105,TRAVEL_AND_LOCAL,Voyage et Local
3321,TOOLS,TOOLS
3363,PERSONALIZATION,PERSONALIZATION
3453,PRODUCTIVITY,Productividad
3607,PARENTING,PARENTING


=====================================
# Step 8: Calculate Monthly Cumulative Installs
=====================================

Group the filtered dataset by Month and Category,
calculate the total installs for each month,
and compute cumulative installs required
for the stacked area chart.

In [21]:
# Total installs by Month and Category
monthly_installs = (
    filtered_data
    .groupby(["Month", "Category_Display"])["Installs"]
    .sum()
    .unstack(fill_value=0)
)

# Arrange months in calendar order
month_order = [
    "Jan","Feb","Mar","Apr","May","Jun",
    "Jul","Aug","Sep","Oct","Nov","Dec"
]

monthly_installs = monthly_installs.reindex(month_order)

# Replace missing values with 0
monthly_installs = monthly_installs.fillna(0)

# Calculate cumulative installs
cumulative_installs = monthly_installs.cumsum()

display(cumulative_installs)

Category_Display,PARENTING,PERSONALIZATION,Productividad,TOOLS,Voyage et Local,写真
Month,,,,,,
Jan,0.0,10000000.0,0.0,1000000.0,0.0,10000000.0
Feb,0.0,20000000.0,100000.0,1000000.0,0.0,10000000.0
Mar,100000.0,20000000.0,100000.0,1000000.0,0.0,62000000.0
Apr,100000.0,20000000.0,100000.0,1000000.0,5000000.0,62000000.0
May,10100000.0,20000000.0,100000.0,11000000.0,5000000.0,73000000.0
Jun,10100000.0,30000000.0,15100000.0,11000000.0,11000000.0,154000000.0
Jul,10700000.0,41500000.0,181100000.0,233600000.0,42100000.0,427000000.0
Aug,10700000.0,61500000.0,786100000.0,353650000.0,77200000.0,953500000.0
Sep,10700000.0,62500000.0,786100000.0,353650000.0,77200000.0,953500000.0


=====================================
# Step 9: Identify Month-over-Month Growth
=====================================

Calculate the month-over-month percentage increase in
installs for each category. Months where installs increase
by more than 25% will be highlighted in the visualization.

In [22]:
# Calculate Month-over-Month Growth (%)
growth = monthly_installs.pct_change() * 100

print("Month-over-Month Growth (%)")
display(growth.round(2))

Month-over-Month Growth (%)


Category_Display,PARENTING,PERSONALIZATION,Productividad,TOOLS,Voyage et Local,写真
Month,,,,,,
Jan,NaN,NaN,NaN,NaN,NaN,NaN
Feb,NaN,0.00,inf,-100.00,NaN,-100.00
Mar,inf,-100.00,-100.00,NaN,NaN,inf
Apr,-100.0,NaN,NaN,NaN,inf,-100.00
May,inf,NaN,NaN,inf,-100.00,inf
Jun,-100.0,inf,inf,-100.00,inf,636.36
Jul,inf,15.00,1006.67,inf,418.33,237.04
Aug,-100.0,73.91,264.46,-46.07,12.86,92.86
Sep,NaN,-95.00,-100.00,-100.00,-100.00,-100.00


=====================================
# Step 10: Check IST Time
=====================================

Verify the current Indian Standard Time (IST).

The visualization should only be displayed
between 4:00 PM IST and 6:00 PM IST.

In [23]:
# Current IST Time
current_time = datetime.now(ZoneInfo("Asia/Kolkata")).time()

start_time = datetime.strptime("16:00", "%H:%M").time()
end_time = datetime.strptime("18:00", "%H:%M").time()

print("Current IST Time:", current_time)

Current IST Time: 00:47:19.946540


=====================================
# Step 11: Create Stacked Area Chart
=====================================

Create a stacked area chart showing cumulative installs
over time for each selected category.

Visualization Features:

• X-axis → Month

• Y-axis → Cumulative Installs

• Separate color band for each category

• Highlight months where installs increased
  by more than 25% for any category

• Display chart only between
  4 PM IST and 6 PM IST

In [24]:
if start_time <= current_time <= end_time:

    plt.figure(figsize=(15,8))

    # Plot stacked area chart
    plt.stackplot(
        cumulative_installs.index,
        cumulative_installs.T.values,
        labels=cumulative_installs.columns,
        alpha=0.8
    )

    # Highlight months where any category grew by >25%
    for month in growth.index:

        if (growth.loc[month] > 25).any():

            plt.axvspan(
                month,
                month,
                color="yellow",
                alpha=0.20
            )

    plt.title(
        "Cumulative Installs Over Time by App Category",
        fontsize=16,
        fontweight="bold"
    )

    plt.xlabel("Month", fontsize=12)

    plt.ylabel("Cumulative Installs", fontsize=12)

    plt.legend(
        title="Category",
        loc="upper left",
        bbox_to_anchor=(1.02,1)
    )

    plt.grid(alpha=0.3)

    plt.tight_layout()

    plt.show()

else:

    print("Stacked Area Chart is available only between 4 PM IST and 6 PM IST.")

Stacked Area Chart is available only between 4 PM IST and 6 PM IST.
